# Exploración de viajes de EcoBici

En este notebook consultamos la tabla `clean.ecobici`, creada en la base `data/db/ecobici.duckdb` con `pipelines/etl-ecobici.sh`.

## 1. Conexión directa con DuckDB

Primero, la forma "a mano": abrir la base con `duckdb.connect` y ejecutar la consulta dentro del bloque `with`.

In [1]:
import duckdb
import pandas as pd

with duckdb.connect("../data/db/ecobici.duckdb", read_only=True) as con:
    resultado = con.sql("""
        SELECT
            genero,
            COUNT(*) AS viajes,
            AVG(edad) AS edad_promedio
        FROM clean.ecobici
        GROUP BY genero
    """)

    resultado.show()

┌─────────┬────────┬───────────────────┐
│ genero  │ viajes │   edad_promedio   │
│ varchar │ int64  │      double       │
├─────────┼────────┼───────────────────┤
│ F       │ 368248 │ 32.12637950511612 │
│ M       │ 988032 │ 34.40908290419743 │
└─────────┴────────┴───────────────────┘



## 2. Función `myquery`

La función `myquery`, definida en `src/progra/tools/mydb.py`, hace lo mismo en una línea: recibe SQL y devuelve un DataFrame.

In [2]:
from progra.db.mydb import myquery

In [4]:
df = myquery("SELECT * FROM clean.ecobici limit 10")
df.head()

,genero,edad,bici,estacion_retiro,fecha_retiro,estacion_arribo,fecha_arribo
0,M,28,2231,156,2014-01-01 00:01:16.037,35,2014-01-01 00:10:18.000
1,M,40,504,112,2014-01-01 00:03:23.223,98,2014-01-01 00:28:18.000
2,F,28,1764,13,2014-01-01 00:03:37.937,2,2014-01-01 00:10:24.000
3,F,26,3749,64,2014-01-01 00:09:17.140,38,2014-01-01 00:16:06.000
4,M,59,2754,33,2014-01-01 00:09:34.520,33,2014-01-01 00:17:55.643


## 3. Análisis bivariado

Agregamos en SQL y después graficamos solo el resultado. Las tres gráficas usan el mismo estilo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="white", context="notebook")
DIAS = {1: "Lun", 2: "Mar", 3: "Mié", 4: "Jue", 5: "Vie", 6: "Sáb", 7: "Dom"}

### 3.1 ¿Cuándo se usan más las bicis? Día de la semana × hora

In [ ]:
viajes_hora = myquery("""
    SELECT
        EXTRACT(ISODOW FROM fecha_retiro) AS dia_num,
        EXTRACT(HOUR FROM fecha_retiro) AS hora,
        COUNT(*) AS viajes
    FROM clean.ecobici
    GROUP BY dia_num, hora
""")

tabla_hora = viajes_hora.pivot(index="dia_num", columns="hora", values="viajes")
tabla_hora.index = tabla_hora.index.map(DIAS)

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(tabla_hora, cmap="rocket_r", ax=ax, cbar_kws={"label": "Viajes"})
ax.set_title("Viajes de retiro por día de la semana y hora (2014)",
             loc="left", fontsize=14, weight="bold")
ax.set_xlabel("Hora del día")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

### 3.2 Origen y destino entre las estaciones principales

Tomamos las 15 estaciones con más retiros y contamos cuántos viajes van de cada estación de retiro a cada estación de arribo. Usamos escala logarítmica porque hay pares con muchos viajes y otros con pocos. Las celdas vacías son pares sin viajes entre estas estaciones.

In [ ]:
origen_destino = myquery("""
    WITH top_estaciones AS (
        SELECT estacion_retiro
        FROM clean.ecobici
        GROUP BY estacion_retiro
        ORDER BY COUNT(*) DESC
        LIMIT 15
    )
    SELECT
        estacion_retiro,
        estacion_arribo,
        COUNT(*) AS viajes
    FROM clean.ecobici
    WHERE estacion_retiro IN (SELECT estacion_retiro FROM top_estaciones)
      AND estacion_arribo IN (SELECT estacion_retiro FROM top_estaciones)
    GROUP BY estacion_retiro, estacion_arribo
""")

matriz = origen_destino.pivot(index="estacion_retiro",
                              columns="estacion_arribo",
                              values="viajes")

fig, ax = plt.subplots(figsize=(9, 8))
sns.heatmap(np.log10(matriz), cmap="viridis", ax=ax,
            cbar_kws={"label": "log10(viajes)"})
ax.set_title("Viajes entre las 15 estaciones con más retiros",
             loc="left", fontsize=14, weight="bold")
ax.set_xlabel("Estación de arribo")
ax.set_ylabel("Estación de retiro")
plt.tight_layout()
plt.show()